# Getting started with maps and light curves

Intrinsic driving belongs to the source. A Kerr disk accepts an optional `driving_signal`, while explicit `ModulatedSource` wrapping applies multiplicative brightness variability. Neither `MicrolensingSystem` nor `MultiImageSystem` accepts a separate driver. On `system.light_curve`, omitting `apply_driving_signal` uses the source's driver when present. `False` holds it at its baseline, and `True` requires a configured driver. A source without a driver works normally without any switch.

This getting-started example generates one Q2237+0305 image-B-like microlensing realization with the high-accuracy production IPM configuration. It uses $N=10^7$, $k=2$, true $r=2$, virtual $v=4$, dual-scout normalization, a $16\times16$ local-membership partition with an $8\times8$ complex-Taylor far-field lattice per cell, temporal map batch 49, memory-bounded label sub-batches of 10, and a 10-frame endpoint-union scout refresh. It returns daily light curves with and without intrinsic variability from one shared 25-day dynamic-map sequence, identifies caustics and anchor/gauge center labels at all 147 map epochs, and writes an animated GIF of the moving magnification map and caustic network.

The calculation is intentionally substantial enough to demonstrate the real CUDA/Triton workflow. Install `microcaustics[notebooks,science]` and run on a CUDA GPU for the intended performance. The astrophysical realization is representative rather than a claim to reproduce one uniquely measured stellar field.

## Runtime and output controls
On CUDA, the default values below run the full 10-year, 147-map production calculation. The CPU branch preserves the physical model and cadences but reduces numerical resolution for a portable demonstration. Set `GENERATE_GIF=False` to retain only the light curves and caustic labels. The output directory contains the GIF and compact NPZ light-curve/label product. The full $1024^2\times147$ map cube is never stored.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import torch
from matplotlib import pyplot as plt
from matplotlib.collections import LineCollection
import microcaustics as mc
import microcaustics.plotting as mcp

plt.rcParams.update(mcp.publication_style())
GENERATE_GIF = True
capabilities = mc.RuntimeCapabilities.detect()
use_cuda = capabilities.cuda_available and capabilities.triton_importable
OUTPUT_DIRECTORY = Path(
    'q2237_production_tutorial_output'
    if use_cuda else '.q2237_cpu_tutorial_output'
)
OUTPUT_DIRECTORY.mkdir(exist_ok=True)
runtime_config = mc.RuntimeConfig(
    profiling="detailed",  # opt in to synchronized component timings
    device='cuda' if use_cuda else 'cpu',
    backend='triton' if use_cuda else 'auto',
    dtype=torch.float32,
    strict_backend=use_cuda,
    memory_fraction=0.95,
)
runtime = mc.resolve_runtime(runtime_config)
print(mcp.runtime_description())
print('Selected device/backend:', runtime.device, runtime.backend.value)
if not use_cuda:
    print('CUDA/Triton unavailable: using reduced numerical resolution.')


## Q2237 image-B-like macro lens and moving stellar field

The local convergence, shear orientation, and redshifts define this example. Stars follow a Salpeter mass function. The stellar velocity dispersion is 170 km/s per component in the lens rest frame. Lens and source peculiar velocities use the redshift-dependent 235 km/s normalization, and the projected CMB dipole supplies the observer motion. The package converts all components to observer-frame angular velocities.

We hold the bulk-velocity draw at seed 0 and choose stellar seed 11 to illustrate a dense caustic network and source-center crossings during the ten-year sequence. This is a deliberately selected example, not an estimate of how often crossings occur. Changing the kinematics seed gives a different bulk-velocity draw.


In [ ]:
lens_redshift, source_redshift = 0.0395, 1.695
macro = mc.MacroLens(
    convergence=0.391,
    shear=0.391,
    shear_angle_deg=141.73,
    smooth_matter_fraction=0.0,
)
population = mc.StellarPopulation.salpeter(
    mean_mass_solar=0.3,
    mass_ratio=100.0,
    kinematics=mc.SkyProjectedKinematics(
        ra_deg=340.126125, dec_deg=3.358611,
        stellar_dispersion_km_s=170.0,
        peculiar_velocity_dispersion_km_s=235.0,
        include_cmb_dipole=True,
        seed=0,  # hold the bulk velocity draw fixed while changing stellar seeds
    ),
)
print('The source model below will set the source field; the complete circular stellar aperture is derived from it.')


## Full-Kerr multiband continuum source and intrinsic driver
A primary-image Kerr transfer supplies light bending and frequency shifts for the steady Novikov--Thorne disk. A positive lognormal broken-power-law driver is sampled daily and heats an axial lamppost, producing a wavelength-dependent delayed thermal response. The microlensing-only comparison is requested together with the driven curve through the same map and label stream; it cannot be obtained by dividing the variable flux by the driver amplitude.


In [ ]:
source_resolution = 1024 if use_cuda else 128
label_resolution = 8192 if use_cuda else 512
ray_count = 10_000_000 if use_cuda else 262_144
black_hole_mass_solar = 10.0**9.08
spin = 0.74
inclination_deg = 10.0
position_angle_deg = 0.0
map_times = torch.arange(0.0, 3650.0 + 12.5, 25.0)
flux_times = torch.arange(0.0, 3650.0 + 0.5, 1.0)
driver = mc.broken_power_law_driving_signal(
    cadence_days=1.0,
    max_duration_days=3650.0,
    history_days=1000.0,
    break_timescale_days=200.0,
    alpha_L=1.0,
    alpha_R=3.0,
    standard_deviation=0.10,
    seed=0,
)
disk_model = mc.KerrDiskModel(
    black_hole_mass_solar=black_hole_mass_solar,
    eddington_ratio=0.34,
    bands_angstrom={"u": 3671.0, "g": 4827.0, "r": 6223.0,
                     "i": 7546.0, "z": 8691.0, "y": 9712.0},
    spin=spin,
    inclination_deg=inclination_deg,
    position_angle_deg=position_angle_deg,
    lamp_fraction=0.1,
    corona_height_above_isco_rg=20.0,
    driving_signal=driver,
    source_grid_shape=source_resolution,
    enclosed_flux_fraction=0.999,
    source_margin=1.05,
    compile_solver=use_cuda,
    lamppost_nalpha=1024 if use_cuda else 256,
    lamppost_radial_bins=512 if use_cuda else 128,
)
# Reuse this small factory whenever a new independent stellar seed is needed.
# Compatible kernels stay compiled for the life of this Python process.
def make_system(seed):
    return mc.MicrolensingSystem(
        lens_redshift=lens_redshift,
        source_redshift=source_redshift,
        H0=70.0,
        Om0=0.3,
        macro=macro,
        source=disk_model,
        stellar_population=population,
        duration_days=3650.0,
        light_loss=0.01,
        safety_scale=1.5,
        seed=seed,
        runtime=runtime_config,
        caustic_grid_shape=label_resolution,
    )

example_seed = 11  # caustic-rich example shared with the reverberation tutorial
system = make_system(example_seed)
realization = system.realize()
distances = system.distances
source_grid = realization.source_grid
variable_disk = realization.source
static_disk = variable_disk.at_driver_mean()
geometry = variable_disk.geometry
gravitational_radius_m = float(mc.gravitational_radius_m(black_hole_mass_solar))
disk_outer_rg = disk_model.support_radius_m(distances) / gravitational_radius_m
half_size_rg = disk_outer_rg
half_size_m = disk_outer_rg * gravitational_radius_m
print('Map epochs:', len(map_times), 'daily flux epochs:', len(flux_times))
print('Full-Kerr hit fraction:', float(static_disk.transfer.hit.float().mean().detach().cpu()))

# Preview the live full-GR reverberating disk. The shared helper accumulates
# temporal statistics in small batches and selects the most variable epoch.
preview_source = variable_disk
preview_times = torch.linspace(
    0.0, 3650.0, 24, dtype=torch.float64, device=runtime.device,
)
disk_summary = mcp.standardize_source_over_time(
    preview_source, preview_times,
    batch_size=4, dtype=torch.float64, device=runtime.device,
    representative='most-variable',
)
disk_figure, disk_axes = mcp.plot_standardized_source_bands(
    disk_summary, source_grid, geometry.band_names,
    scale_bar_uas=1.0, figsize=(15.6, 2.7),
)
disk_figure.savefig(
    OUTPUT_DIRECTORY / 'q2237_relativistic_disk.png',
    dpi=220, bbox_inches='tight', facecolor='white',
)

simulation = realization.simulation
stars = realization.stars
lens_region = realization.lens_region
lens_grid = realization.lens_grid
source_grid = realization.source_grid
aperture_radius_uas = realization.stellar_aperture.radius_uas
realized_circular_kappa = float(stars.einstein_radius_uas.square().sum() / aperture_radius_uas**2)
print('Stars:', len(stars), 'realized circular kappa_*:', f'{realized_circular_kappa:.4f}')
print('Derived source/lens fields [microarcsec]:', source_grid.field_of_view_uas, lens_region.field_of_view_uas)

# Report the physical components and actual observer-frame angular velocities.
motion = population.kinematics.resolved_metadata(distances)
print('Motion components:', motion)
print('Sampled stellar motion:', system.metadata()['stellar_motion'])
print('Ten-year bulk displacement [microarcsec]:',
      np.asarray(motion['bulk_velocity_uas_per_day']) * 3650.0)


## Production dynamic IPM and shared multirate light curve
Only the slow magnification field is generated every 25 days. The source and light curve are evaluated daily by interpolating the two map--source contractions that bracket each epoch. Interpolated maps are never materialized. Analytic detA, marching squares, mapped caustics, and nine-anchor/nine-gauge source-center labels share the same per-epoch local-exact/Taylor far-field states. All 147 map epochs are retained at display resolution for the GIF, while the full-resolution map cube is never stored.

In [ ]:
method = mc.production_ipm_config(rays=ray_count)
schedule = mc.production_dynamic_config()

caustic_config = mc.CausticConfig(
    far_field_approx=method.far_field_approx,
    temporal_batch_size=10,
    jacobian_chunk_size=1_048_576,
    anchor_count=9,
    gauge_count=9,
)

gif_maps, gif_times, gif_caustics = [], [], []
gif_center_magnifications = []
gif_center_labels, gif_crossings, gif_center_distances = [], [], []
def retain_gif_frame(index, labeled_frame):
    if GENERATE_GIF:
        magnification_map = labeled_frame.magnification_map
        caustic_frame = labeled_frame.caustics
        # Retain the calculated 1024-square map without pixel averaging.
        gif_maps.append(magnification_map.values.detach().cpu())
        gif_times.append(float(magnification_map.time_days))
        center_row = magnification_map.values.shape[0] // 2
        center_column = magnification_map.values.shape[1] // 2
        gif_center_magnifications.append(
            float(magnification_map.values[center_row, center_column].detach().cpu())
        )
        caustics = caustic_frame.caustics
        valid = caustics.invalid_segment_mask
        valid = (
            torch.ones(caustics.segment_count, device=caustics.caustic_segments_uas.device, dtype=torch.bool)
            if valid is None else ~valid
        )
        gif_caustics.append(
            caustics.caustic_segments_uas[valid].detach().cpu().numpy()
        )
        gif_center_labels.append(int(caustic_frame.labels.center_label))
        gif_crossings.append(bool(caustic_frame.labels.center_crossing))
        gif_center_distances.append(float(caustic_frame.labels.center_distance_uas))

labeled_result = system.light_curve(
    map_times,
    flux_times_days=flux_times,
    include_labels=True,
    include_microlensing_only=True,
    method=method,
    schedule=schedule,
    caustics=caustic_config,
    map_observer=retain_gif_frame,
)
variable_curve = labeled_result
print('Actual rasterizer:', variable_curve.metadata['dynamic_map_metadata'].get('rasterizer'))
print('Single end-to-end labeled workflow + GIF capture [s]:', variable_curve.timing.delivered_seconds)
print('Timing components [s]:', dict(variable_curve.timing.component_seconds))
print('This is not warmed LC-only timing: it includes caustics, labels, and synchronous CPU GIF-frame capture.')
crossing_times = labeled_result.labels.times_days[labeled_result.labels.crossing_events]
print('Center-label crossings:', int(crossing_times.numel()))
print('Crossing times [days]:', [round(float(value), 1) for value in crossing_times])
print('Retained GIF frames:', len(gif_maps), 'of', len(map_times))
print('Mean-driver comparison shares the map and label calculation')
assert variable_curve.times_days.numel() == 3651
assert variable_curve.microlensing_only_flux.shape == variable_curve.flux.shape
driver_values = driver.amplitudes(
    flux_times, bands=1, dtype=variable_curve.flux.dtype,
    device=variable_curve.flux.device,
)[:, 0].detach().cpu()
print(
    'Intrinsic driver: positive lognormal broken PSD; min/mean/max =',
    *(f'{value:.4f}' for value in (driver_values.min(), driver_values.mean(), driver_values.max())),
)
print('Both thermal light curves use one-day source sampling.')


## The actual source scout and biquadratic IPM cell
This cell generates the method schematic directly from the sky-frame Q2237 image-B realization constructed above. The physical shear and disk position angles remain separate; the disk is not manually rotated to absorb the shear. The left panel runs the actual $k=2$ source-directed scout. The right panel traces the true $r=2$ nodes and evaluates the biquadratic representation on the virtual $v=4$ lattice. The plotting conventions are unchanged, and no saved diagnostic arrays or pre-rendered image are loaded.


In [ ]:
from IPython.display import Image, display

# Generate the scout mask and representative mapped cell from the live
# Q2237 B realization above. The renderer only controls presentation.
paper_ipm_paths = mcp.render_live_paper_ipm_schematic(
    simulation,
    lens_region,
    source_grid,
    method,
    OUTPUT_DIRECTORY,
    time_days=0.0,
    output_prefix="q2237_image_b_scout_and_ipm",
)
display(Image(filename=str(paper_ipm_paths[0])))


## Light curves and labels at the source center
Both light curves contain the same finite-source microlensing realization. Their only difference is the known intrinsic driving signal. The second figure presents the source-center diagnostics separately from the GIF. They are center magnification, the caustic-crossing label, and the distance to the nearest caustic.

In [ ]:
band = "i"
band_index = geometry.band_names.index(band)
# The GR disk returns physical flux density in Jy, so AB magnitudes follow
# directly from the standard 3631 Jy definition without a fitted reference.

figure, ax = plt.subplots(figsize=(8.4, 4.0))
ax.plot(
    flux_times.numpy(),
    variable_curve.microlensing_only_magnitude[:, band_index].detach().cpu().numpy(),
    label="Microlensing only",
    linewidth=1.8,
)
ax.plot(
    flux_times.numpy(),
    mc.flux_to_magnitude(variable_curve.flux[:, band_index]).detach().cpu().numpy(),
    label="Microlensing + intrinsic variability",
    linewidth=1.2,
    alpha=0.9,
)
for crossing_time in labeled_result.labels.times_days[labeled_result.labels.crossing_events]:
    ax.axvline(float(crossing_time), color="0.35", linewidth=0.8, alpha=0.35)
ax.set(xlabel="Observer time [days]", ylabel="brightness [mag]")
ax.invert_yaxis()
ax.legend(frameon=True)
mcp.finish_axis(ax)
figure.tight_layout()

label_figure, label_axes = plt.subplots(
    3, 1, figsize=(8.4, 5.3), sharex=True, gridspec_kw={"hspace": 0.08}
)
label_axes[0].set_title("Labels at source center")
label_axes[0].plot(
    gif_times,
    np.log10(np.clip(gif_center_magnifications, 1.0e-12, None)),
    color="black",
    linewidth=1.1,
)
label_axes[0].set_ylabel(r"$\log_{10}\mu$")
label_axes[1].step(
    gif_times,
    np.asarray(gif_crossings, dtype=float),
    where="pre",
    color="black",
    linewidth=1.1,
)
label_axes[1].set(ylim=(-0.05, 1.05), yticks=(0, 1), ylabel="crossing")
label_axes[2].plot(gif_times, gif_center_distances, color="black", linewidth=1.1)
label_axes[2].set(
    ylim=(0.0, None),
    xlabel="Observer time [days]",
    ylabel=r"$d_{\rm caustic}$ [$\mu$as]",
)
for label_axis in label_axes:
    mcp.finish_axis(label_axis)
for label_axis in label_axes[:-1]:
    label_axis.tick_params(labelbottom=False)
label_figure.align_ylabels(label_axes)
label_figure.subplots_adjust(left=0.13, right=0.98, bottom=0.12, top=0.92)
label_figure.savefig(OUTPUT_DIRECTORY / "q2237_image_b_source_center_labels.png", dpi=180)

np.savez_compressed(
    OUTPUT_DIRECTORY / "q2237_image_b_light_curves.npz",
    times_days=flux_times.numpy(),
    flux_with_variability=variable_curve.flux.detach().cpu().numpy(),
    flux_without_variability=variable_curve.microlensing_only_flux.detach().cpu().numpy(),
    unlensed_with_variability=variable_curve.unlensed_flux.detach().cpu().numpy(),
    band_names=np.asarray(variable_curve.band_names),
    map_times_days=labeled_result.labels.times_days.numpy(),
    center_labels=labeled_result.labels.crossing_labels.numpy(),
    center_crossings=labeled_result.labels.crossing_events.numpy(),
    center_distance_uas=labeled_result.labels.center_distances_uas.numpy(),
    metadata_json=np.asarray(json.dumps(dict(variable_curve.metadata), default=str)),
)


## Animate the moving magnification map

The GIF displays $\log_{10}\mu$ over the complete source field. All 147 frames use the same global minimum and maximum, colorbar, and palette. The original 1024-square maps are retained without pixel averaging. The animation canvas scales their display to fit the page. The palette explicitly preserves the magma gradient and blue caustic lines without error-diffusion dithering. The source-center labels are plotted separately above.


In [ ]:
if GENERATE_GIF:
    log_maps = torch.log10(torch.stack(gif_maps).clamp_min(1.0e-2)).numpy()
    finite = log_maps[np.isfinite(log_maps)]
    vmin, vmax = float(finite.min()), float(finite.max())
    figure, ax = plt.subplots(figsize=(5.75, 5.0))
    xmin, xmax, ymin, ymax = source_grid.bounds_uas
    image = ax.imshow(
        log_maps[0],
        origin="lower",
        extent=(xmin, xmax, ymin, ymax),
        cmap="magma",
        vmin=vmin,
        vmax=vmax,
        interpolation="bilinear",
    )
    time_text = ax.text(
        0.08, 0.91, "", transform=ax.transAxes, ha="left", va="top",
        color="white", fontsize=13, clip_on=False,
        bbox={"facecolor": "black", "edgecolor": "none", "alpha": 0.35, "pad": 2.5},
    )
    caustic_lines = LineCollection(
        gif_caustics[0], colors="deepskyblue", linewidths=0.55, alpha=0.9, zorder=8
    )
    ax.add_collection(caustic_lines, autolim=False)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    mcp.add_scale_bar(
        ax, 1.0, label=r"1 $\mu$as", location="lower left", color="white",
        font_size=11, thickness_fraction=0.007, borderpad=2.0, pad=0.35,
    )
    mcp.hide_image_axes(ax)
    ax.set_box_aspect(1)
    mcp.panel_colorbar(
        figure, ax, image, label=r"$\log_{{10}}\mu$", size="2.6%", pad=0.035
    )
    # Reserve explicit room for the colorbar label. This prevents the final
    # pixels from being clipped by notebook or GIF renderers.
    figure.subplots_adjust(left=0.035, right=0.82, bottom=0.045, top=0.98)

    rendered_frames = []
    for frame_index in range(len(gif_maps)):
        image.set_data(log_maps[frame_index])
        caustic_lines.set_segments(gif_caustics[frame_index])
        time_text.set_text(f"t = {gif_times[frame_index]:.0f} days")
        figure.canvas.draw()
        rendered_frames.append(np.asarray(figure.canvas.buffer_rgba())[..., :3].copy())

    gif_path = OUTPUT_DIRECTORY / "q2237_image_b_moving_magnification_map.gif"
    # Keep the full gradient and thin blue caustics in the GIF palette.
    palette_rgb = np.vstack((
        (plt.get_cmap("magma")(np.linspace(0, 1, 224))[:, :3] * 255).astype(np.uint8),
        np.repeat(np.linspace(0, 255, 30, dtype=np.uint8)[:, None], 3, axis=1),
        np.array([[0, 191, 255], [255, 255, 255]], dtype=np.uint8),
    ))
    mcp.save_fixed_palette_gif(
        rendered_frames, gif_path, fps=8, dither=False, palette_rgb=palette_rgb,
    )
    plt.close(figure)
    print("Wrote", gif_path, "with one fixed colormap palette and no error-diffusion flicker")
    try:
        from IPython.display import Image, display
        display(Image(filename=str(gif_path)))
    except ImportError:
        pass


## Generate another realization

Create another system with a different seed and run it in the same Python process. The physical stellar field changes, while compatible compiled kernels are reused automatically.


In [ ]:
second_system = make_system(1)
second_map = second_system.magnification_map(time_days=0.0, method=method)
print("Seed 1 stars:", len(second_system.realize().stars))
print(
    "Seed 1 warmed map time [s]:",
    f"{second_map.timing.delivered_seconds:.3f}",
)
print("The compatible kernels were reused automatically.")
